In [ ]:
from pyspark.sql import functions as F

# DATA LAKE LOCATION
storage = "YOURLOCATION"

lake = (
    f"abfss://sales@{storage}.dfs.core.windows.net"
)

# 1. EXTRACT - read the JSON files
carts = (
    spark.read
    .option("multiLine", "true")
    .json(f"{lake}/raw/carts.json")
)

StatementMeta(sparkSalesLab, 3, 2, Finished, Available, Finished, False)

In [23]:
# 1. EXTRACT - read the JSON files
carts = (
    spark.read
    .option("multiLine", "true")
    .json(f"{lake}/raw/carts.json")
)

products = (
    spark.read
    .option("multiLine", "true")
    .json(f"{lake}/raw/products.json")
)

print("Carts:", carts.count())
print("Products:", products.count())

StatementMeta(sparkSalesLab, 3, 3, Finished, Available, Finished, False)

Carts: 7
Products: 20


In [24]:
# 2. TRANSFORM - flatten nested cart items
cart_lines = (
    carts
    .dropDuplicates(["id"])
    .select(
        F.col("id").cast("int").alias("CartId"),
        F.col("userId").cast("int").alias("UserId"),
        F.to_date("date").alias("CartDate"),
        F.explode("products").alias("Item")
    )
    .select(
        "CartId", "UserId", "CartDate",
        F.col("Item.productId").cast("int")
            .alias("ProductId"),
        F.col("Item.quantity").cast("int")
            .alias("Quantity")
    )
)

api_cart_lines = cart_lines

StatementMeta(sparkSalesLab, 3, 4, Finished, Available, Finished, False)

In [25]:

import random
from datetime import date, timedelta

# Use a fixed seed for reproducible test data
rng = random.Random(42)

# Get valid product IDs from the real API catalogue
product_ids = [
    int(row["id"])
    for row in products.select("id").distinct().collect()
    if row["id"] is not None
]

if not product_ids:
    raise ValueError("No product IDs were loaded")

synthetic_rows = []

# Generate 5,000 simulated shopping carts
for cart_num in range(1, 5001):

    cart_id = 100000 + cart_num
    user_id = rng.randint(1000, 9999)

    cart_date = date(2025, 1, 1) + timedelta(
        days=rng.randint(0, 364)
    )

    number_of_products = rng.randint(
        1, min(4, len(product_ids))
    )

    selected_products = rng.sample(
        product_ids, number_of_products
    )

    for product_id in selected_products:
        quantity = rng.randint(1, 3)

        synthetic_rows.append((
            cart_id,
            user_id,
            cart_date,
            product_id,
            quantity
        ))

# Create a Spark DataFrame
synthetic_lines = spark.createDataFrame(
    synthetic_rows,
    schema="""
        CartId INT,
        UserId INT,
        CartDate DATE,
        ProductId INT,
        Quantity INT
    """
)

# Label the source of each record
original_lines = api_cart_lines.withColumn(
    "DataOrigin", F.lit("API sample")
)

synthetic_lines = synthetic_lines.withColumn(
    "DataOrigin", F.lit("Synthetic")
)

# Combine original and generated records
cart_lines = original_lines.unionByName(
    synthetic_lines
)

print("Total cart lines:", cart_lines.count())
print("Total unique carts:",
      cart_lines.select("CartId").distinct().count())

display(cart_lines.limit(10))


StatementMeta(sparkSalesLab, 3, 5, Finished, Available, Finished, False)

Total cart lines: 12557
Total unique carts: 5007


SynapseWidget(Synapse.DataFrame, 6dad7bf0-0713-4b3c-9d4b-b5f1dd191994)

In [26]:
# 3. Prepare product dimension
dim_product = (
    products
    .select(
        F.col("id").cast("int").alias("ProductId"),
        F.col("title").alias("ProductName"),
        F.col("category").alias("Category"),
        F.col("price").cast("decimal(12,2)")
            .alias("UnitPrice")
    )
    .dropDuplicates(["ProductId"])
    .filter(F.col("ProductId").isNotNull())
    .filter(F.col("UnitPrice") >= 0)
)

StatementMeta(sparkSalesLab, 3, 6, Finished, Available, Finished, False)

In [27]:
# 4. Join carts and products
fact_cart_lines = (
    cart_lines
    .join(
        dim_product.select("ProductId", "UnitPrice"),
        on="ProductId",
        how="inner"
    )
    .filter(
        (F.col("Quantity") > 0) &
        F.col("CartDate").isNotNull()
    )
    .withColumn(
        "LineBasketValue",
        (F.col("Quantity") * F.col("UnitPrice"))
        .cast("decimal(18,2)")
    )
)

StatementMeta(sparkSalesLab, 3, 7, Finished, Available, Finished, False)

In [28]:
# 5. Create date dimension
dim_date = (
    fact_cart_lines
    .select(F.col("CartDate").alias("Date"))
    .distinct()
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn(
        "MonthName",
        F.date_format("Date", "MMMM")
    )
)

StatementMeta(sparkSalesLab, 3, 8, Finished, Available, Finished, False)

In [29]:
# 6. Basic validation
if fact_cart_lines.count() == 0:
    raise ValueError("No valid cart lines found")

print("Cart lines:", fact_cart_lines.count())

StatementMeta(sparkSalesLab, 3, 9, Finished, Available, Finished, False)

Cart lines: 12557


In [30]:
# 7. LOAD - create curated Parquet files
fact_cart_lines.write.mode("overwrite").parquet(
    f"{lake}/curated/fact_cart_lines"
)

dim_product.write.mode("overwrite").parquet(
    f"{lake}/curated/dim_product"
)

dim_date.write.mode("overwrite").parquet(
    f"{lake}/curated/dim_date"
)

print("Transformation completed successfully")

StatementMeta(sparkSalesLab, 3, 10, Finished, Available, Finished, False)

Transformation completed successfully


In [31]:
display(fact_cart_lines.limit(10))

StatementMeta(sparkSalesLab, 3, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 457e72a0-9620-4d14-b9db-4ffb785a9f97)